# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social page', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'portfolio project page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio project page',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog/zh'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Zhihu', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'Careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/embeddinggemma-2
Updated
8 days ago
•
135k
•
2.27k
jialinyyzz/humanizer
Updated
13 minutes ago
•
65.6k
•
1.64k
Qwen/Qwen-Image-2.1-Turbo
Updated
3 days ago
•
12.7k
•
778
openbmb/MiniCPM-V-4.7-35B-A3B
Updated
about 11 hours ago
•
4.16k
•
633
abenzerps/Qwen-Image-2.1-Uncensored-Q

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/embeddinggemma-2\nUpdated\n8 days ago\n•\n135k\n•\n2.27k\njialinyyzz/humanizer\nUpdated\n14 minutes ago\n•\n6

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face  
*The AI community building the future.*

---

## About Hugging Face  
Hugging Face is the premier collaboration platform for the machine learning (ML) community, hosting a vast ecosystem where developers, researchers, and enterprises come together to create, share, and innovate on AI models, datasets, and applications. As the "Home of Machine Learning," Hugging Face empowers users to build the future of artificial intelligence through open collaboration and cutting-edge tools.

---

## What We Offer

### Extensive Model Hub  
- Access to **2 million+ ML models** spanning various AI tasks, updated frequently by community and corporate contributors.  
- Trending models include transformers for embedding, humanization, image generation, and more.

### Datasets  
- Explore over **500,000 datasets** curated for training and evaluating ML algorithms, ranging from social media video content to Reinforcement Learning environments.

### Spaces & Applications  
- Host and interact with **1 million+ AI applications** in Spaces, enabling users to deploy and demo models with real-world use cases such as video generation, face swapping, and decision indexing.

### Enterprise Solutions  
- Support for enterprise needs with premium services like **Hugging Face PRO, enterprise support**, customized inference endpoints, and storage buckets for scalable ML deployment.

---

## Community & Collaboration

- A vibrant, open community platform where ML practitioners collaborate on shared projects and contribute models, datasets, and innovations.  
- Interactive forums like Discord and GitHub facilitate ongoing dialogue, knowledge exchange, and co-creation.  
- Daily updated blog posts, research paper digests, and hardware insights keep the community informed and engaged.

---

## Company Culture

- **Open collaboration:** Hugging Face thrives on transparency and community involvement, fostering an environment where sharing knowledge accelerates AI progress globally.  
- **Innovation-driven:** Constantly pushing the boundaries of AI through community contributions and in-house research.  
- **Inclusive & welcoming:** Encourages diversity in skills and backgrounds to build a vibrant ecosystem for machine learning advancements.  
- **User-focused:** Prioritizes tools and services that empower both individual developers and large organizations to leverage AI effectively.

---

## Customers & Partners

- Serves a broad spectrum—from individual researchers to Fortune 500 enterprises—enabling seamless AI model sharing, deployment, and integration.  
- Trusted partner for deploying scalable ML infrastructure with solutions tailored for production-critical environments.

---

## Career & Opportunities

Join the team at Hugging Face and be part of shaping the future of AI! The company looks for passionate individuals eager to:

- Develop open-source AI tools and frameworks.  
- Enhance ML infrastructure at scale.  
- Engage directly with a global community of AI researchers and developers.  
- Drive innovation with cutting-edge machine learning models and applications.

Explore current job openings and learn more about working at Hugging Face on their website.

---

## Connect & Learn More

- Explore models and datasets: [huggingface.co/models](https://huggingface.co/models)  
- Engage with community: Discord & GitHub  
- Blog & Research papers for latest AI trends  
- Enterprise solutions tailored to your needs

---

**Hugging Face**  
*Empowering the AI community to create the future of machine learning.*  
Join us, contribute, and innovate together!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Us

Hugging Face is a vibrant platform and community dedicated to advancing the field of machine learning and artificial intelligence. It serves as a collaborative hub where developers, researchers, and organizations come together to create, share, and innovate with models, datasets, and applications. Known as the home of machine learning, Hugging Face empowers the global AI community to build the future through open collaboration.

---

## What We Offer

- **2 Million+ Models:** Explore a vast and ever-growing repository of machine learning models across various domains and tasks.
- **500,000+ Datasets:** Access and contribute to a rich collection of datasets to train, validate, or benchmark AI models.
- **Spaces:** Host and discover AI-powered applications in an interactive environment.
- **AI Applications & Tools:** Browse and use over 1 million AI applications developed by the community.
- **Enterprise Solutions:** Specialized offerings with Hugging Face PRO, enterprise-grade support, inference providers, endpoints, and storage buckets to scale AI in business environments.
- **Learning & Community Resources:** Engage with blogs, forums, Discord channels, daily paper reviews, and GitHub repositories to stay at the cutting edge of AI research and development.

---

## Our Community & Customers

Hugging Face is the quintessential platform for:

- **AI Researchers & Developers:** Collaborating on state-of-the-art models and sharing innovations.
- **Enterprises & Startups:** Leveraging AI and machine learning models at scale with expert support and infrastructure.
- **Educators & Students:** Accessing open resources for learning and experimentation.
- **AI Enthusiasts & Innovators:** Engaging with a broad community sensitive to transparency, reproducibility, and open source collaboration.

The platform fosters a collaborative spirit making it easy to share, improve, and deploy AI solutions globally.

---

## Company Culture

At Hugging Face, collaboration and openness are at the core of everything we do. We believe in building AI that benefits everyone. The company culture emphasizes:

- **Community-Centric Development:** Prioritizing user-driven improvements and open source contributions.
- **Innovation & Learning:** Staying at the forefront of AI research and providing tools to democratize machine learning.
- **Transparency & Inclusivity:** Cultivating a welcoming environment where diverse ideas and backgrounds thrive.
- **Passion for AI:** Driven by a mission to make machine learning accessible and impactful across industries and disciplines.

---

## Careers at Hugging Face

Join a forward-thinking team passionate about shaping the AI landscape.

- Current openings span across engineering, research, product, design, and developer relations.
- Work alongside top AI talent in a dynamic and supportive environment.
- Contribute directly to projects used by millions around the world.
- Be part of a company that values openness, collaboration, and technological excellence.

Explore current job opportunities and grow your AI career with Hugging Face.

---

## Brand & Visual Identity

- The Hugging Face brand is recognized by its vibrant colors:
  - Yellow: #FFD21E
  - Orange: #FF9D00
  - Gray: #6B7280
- Official logos and brand assets are available for partners and contributors to maintain consistent identity across projects.

---

## Connect with Us

- Explore AI apps, models, and datasets at: [huggingface.co](https://huggingface.co)
- Join the conversation on [Discord](https://discord.gg/huggingface), [Forum](https://discuss.huggingface.co), and [GitHub](https://github.com/huggingface)
- Follow our insights and news through the Hugging Face Blog and Daily Papers posts.

---

## Join Hugging Face — Collaborate, Create, and Build the Future of AI.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>